# doc-structure-mcp 模型选型测试（AI Studio）

目标：找到能正确识别红头竖排文字的 OCR 模型。

AI Studio 已预装 PaddlePaddle，无需额外安装。

## 用法
1. 点击右上角「登录」，用百度账号登录
2. 菜单栏「运行」→「运行全部」
3. 上传测试文件时选择 test_scanned_doc.pdf

In [ ]:
# 1. 检查环境
import paddle
print(f'PaddlePaddle: {paddle.__version__}')
print(f'GPU: {paddle.is_compiled_with_cuda()}')
!nvidia-smi | head -5

# 安装 PaddleOCR
!pip install paddleocr==3.6.0 -q
print('PaddleOCR 安装完成')

In [ ]:
# 2. 上传测试文件
from google.colab import files  # AI Studio 兼容
print('请上传 test_scanned_doc.pdf（第1页即可）')
uploaded = files.upload()
import os
image_path = list(uploaded.keys())[0] if uploaded else None
print(f'文件: {image_path}')

In [ ]:
# 3. 测试函数
import cv2, numpy as np, time, json
from paddleocr import PaddleOCR

def test_model(det_model, rec_model, img_path, name='test'):
    """测试一组模型组合"""
    res = {'name': name, 'det': det_model, 'rec': rec_model}
    try:
        t0 = time.time()
        ocr = PaddleOCR(
            text_detection_model_name=det_model,
            text_recognition_model_name=rec_model,
            use_textline_orientation=True, lang='ch',
            text_det_thresh=0.3, text_det_box_thresh=0.5,
        )
        res['load_time'] = round(time.time() - t0, 1)
        
        img = cv2.imread(img_path)
        t0 = time.time()
        result = list(ocr.predict(img))
        res['infer_time'] = round(time.time() - t0, 1)
        
        texts = []
        for r in result:
            if r is None: continue
            tl = r.get('rec_texts',[]) if isinstance(r,dict) else getattr(r,'rec_texts',[])
            sl = r.get('rec_scores',[]) if isinstance(r,dict) else getattr(r,'rec_scores',[])
            for i in range(len(tl)):
                t = tl[i].strip()
                s = float(sl[i]) if i < len(sl) else 0.0
                if t and s > 0.3:
                    texts.append(t)
        res['lines'] = len(texts)
        res['texts'] = texts[:30]
        res['status'] = 'OK'
    except Exception as e:
        res['status'] = f'ERROR: {str(e)[:100]}'
    return res

In [ ]:
# 4. 执行测试矩阵
if image_path:
    tests = [
        ('v1-mobile', 'PP-OCRv5_mobile_det', 'PP-OCRv5_mobile_rec'),
        ('v2-SVTRv2', 'PP-OCRv5_mobile_det', 'ch_SVTRv2_rec'),
        ('v3-RepSVTR', 'PP-OCRv5_mobile_det', 'ch_RepSVTR_rec'),
        ('v4-server', 'PP-OCRv5_server_det', 'PP-OCRv5_server_rec'),
    ]
    
    all_r = []
    for name, det, rec in tests:
        print(f'\n--- {name} ---')
        print(f'  检测: {det}  识别: {rec}')
        r = test_model(det, rec, image_path, name)
        all_r.append(r)
        if r['status'] == 'OK':
            print(f'  耗时: 加载{r["load_time"]}s + 推理{r["infer_time"]}s')
            print(f'  行数: {r["lines"]}')
            for t in r['texts'][:20]:
                print(f'    {t}')
        else:
            print(f'  ERROR: {r["status"]}')
else:
    print('请先上传测试文件')

In [ ]:
# 5. 汇总
print('\n========== 汇总 ==========')
print(f'{"测试":<12} {"状态":<8} {"耗时":<10} {"行数":<6} {"红头关键词"}')
print('-'*60)
keywords = ['财政','税务','人民','银行','国务','档案','标准','民航','铁路','通知']
for r in all_r:
    if r['status'] == 'OK':
        t = f'{r["load_time"]+r["infer_time"]:.1f}s'
        found = [k for k in keywords if any(k in x for x in r['texts'])]
        print(f'{r["name"]:<12} {"OK":<8} {t:<10} {r["lines"]:<6} {found}')
    else:
        print(f'{r["name"]:<12} {"ERR":<8} {"-":<10} {"-":<6} {r["status"][:30]}')

In [ ]:
# 6. 额外测试：PaddleX 3.7.2（PP-OCRv6_medium）
# 取消注释运行
"""
!pip uninstall paddleocr paddlex -y
!pip install paddlex[ocr]==3.7.2 -q

from paddlex import create_pipeline
ocr = create_pipeline(pipeline='OCR')
res = list(ocr.predict(image_path))
for r in res:
    if r:
        texts = getattr(r, 'rec_texts', []) or []
        for t in texts:
            print(t)
"""